# 6.3: Parameter Initialization

In [1]:
import torch
from torch import nn

In [2]:
net = nn.Sequential(nn.LazyLinear(8), nn.ReLU(), nn.LazyLinear(1))
X = torch.rand(size=(2,4))
net(X).shape

c:\Users\Py Torch\.conda\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


torch.Size([2, 1])

## 6.3.1: Built-in Initialization

In [3]:
def init_normal(module):
    if type(module) == nn.Linear:
        nn.init.normal_(module.weight, mean=0, std=0.01)
        nn.init.zeros_(module.bias) # zero out bias

net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([ 0.0030, -0.0046,  0.0027,  0.0178]), tensor(0.))

In [4]:
def init_constant(module):
    if type(module) == nn.Linear:
        nn.init.constant_(module.weight, 1) # initialize to constant
        nn.init.zeros_(module.bias)

net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

Using different initializers for certain blocks:

In [5]:
def init_xavier(module):
    if type(module) == nn.Linear:
        nn.init.xavier_uniform_(module.weight)

def init_42(module):
    if type(module) == nn.Linear:
        nn.init.constant_(module.weight, 42)

net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([-0.4316,  0.0056,  0.4904,  0.2650])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


### Custom Initialization

Example strange distribution: 
$w \sim \begin{cases} U(5, 10) \ \text{with probability }\frac{1}{4} \\ 0 \ \text{with probability }\frac{1}{2} \\ U(-10, -5) \ \text{with probability }\frac{1}{4} \\  \end{cases}$

In [7]:
def my_init(module):
    if type(module) == nn.Linear:
        print("Init", *[(name, param.shape) 
                        for name, param in module.named_parameters()][0])
        nn.init.uniform_(module.weight, -10, 10)
        module.weight.data *= module.weight.data.abs() >= 5
    
net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[-0.0000, -0.0000,  8.6003, -0.0000],
        [ 8.6449, -0.0000, -0.0000, -8.7726]], grad_fn=<SliceBackward0>)

Setting parameters directly:

In [8]:
net[0].weight.data[:] += 1
net[0].weight.data[0,0] = 42
net[0].weight.data[0]

tensor([42.0000,  1.0000,  9.6003,  1.0000])